# Hotel Booking Demand & Cancellation Analysis (EDA)

This notebook executes a complete Exploratory Data Analysis (EDA) on the Hotel Booking Demand dataset. 
It analyzes booking patterns, cancellation drivers, pricing dynamics (ADR), distribution channels, and customer demographics across **City Hotel** and **Resort Hotel** properties.

---

## 1. Environment Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Load cleaned dataset
df = pd.read_csv("hotel_bookings_cleaned.csv")
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Dataset Overview & Data Quality Check

In [ ]:
# Summary Info
print("--- Info ---")
df.info()

print("\n--- Missing Values ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\n--- Summary Statistics ---")
df[["lead_time", "adr", "total_stay_nights", "total_guests", "total_of_special_requests"]].describe()

## 3. High-Level Metrics & Property Comparison

In [ ]:
kpis = df.groupby("hotel").agg(
    Total_Bookings=("is_canceled", "count"),
    Cancellation_Rate=("is_canceled", "mean"),
    Avg_Lead_Time=("lead_time", "mean"),
    Avg_ADR=("adr", "mean"),
    Avg_Stay_Nights=("total_stay_nights", "mean"),
    Repeat_Guest_Rate=("is_repeated_guest", "mean")
).reset_index()

kpis["Cancellation_Rate"] = (kpis["Cancellation_Rate"] * 100).round(2).astype(str) + "%"
kpis["Repeat_Guest_Rate"] = (kpis["Repeat_Guest_Rate"] * 100).round(2).astype(str) + "%"
kpis["Avg_ADR"] = "€" + kpis["Avg_ADR"].round(2).astype(str)
kpis["Avg_Lead_Time"] = kpis["Avg_Lead_Time"].round(1)
kpis["Avg_Stay_Nights"] = kpis["Avg_Stay_Nights"].round(1)

kpis

## 4. Cancellation Rate & Lead Time Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(x="hotel", hue="is_canceled", data=df, palette="Set2", ax=ax)
ax.set_title("Cancellations by Hotel Type (0 = Confirmed, 1 = Canceled)", fontsize=14, fontweight="bold")
ax.set_xlabel("Hotel Type")
ax.set_ylabel("Number of Bookings")
plt.tight_layout()
plt.show()

In [ ]:
# Binning lead times
bins = [0, 14, 60, 120, 700]
labels = ["0-14 Days", "15-60 Days", "61-120 Days", "120+ Days"]
df["lead_time_bucket"] = pd.cut(df["lead_time"], bins=bins, labels=labels, include_lowest=True)

lead_cancel = df.groupby("lead_time_bucket")["is_canceled"].mean().reset_index()
lead_cancel["cancellation_pct"] = lead_cancel["is_canceled"] * 100

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x="lead_time_bucket", y="cancellation_pct", data=lead_cancel, palette="Reds_d", ax=ax)
ax.set_title("Cancellation Rate by Booking Lead Time Window", fontsize=14, fontweight="bold")
ax.set_xlabel("Lead Time Window")
ax.set_ylabel("Cancellation Rate (%)")
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Pricing & Seasonality Trends (ADR)

In [ ]:
ordered_months = ["January", "February", "March", "April", "May", "June", 
                  "July", "August", "September", "October", "November", "December"]
df["arrival_date_month"] = pd.Categorical(df["arrival_date_month"], categories=ordered_months, ordered=True)

plt.figure(figsize=(12, 5))
sns.lineplot(x="arrival_date_month", y="adr", hue="hotel", data=df[df["is_canceled"] == 0], ci=None, marker="o", linewidth=2.5)
plt.title("Average Daily Rate (€) Seasonality Across Months", fontsize=14, fontweight="bold")
plt.xlabel("Arrival Month")
plt.ylabel("Average Daily Rate (€)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Distribution Channels & Guest Origin

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top Market Segments
sns.countplot(x="market_segment", data=df, palette="viridis", order=df["market_segment"].value_counts().index, ax=axes[0])
axes[0].set_title("Bookings by Market Segment", fontsize=12, fontweight="bold")
axes[0].tick_params(axis='x', rotation=30)

# Top 10 Guest Countries
top_10_countries = df["country"].value_counts().head(10).reset_index()
top_10_countries.columns = ["country", "count"]
sns.barplot(x="country", y="count", data=top_10_countries, palette="mako", ax=axes[1])
axes[1].set_title("Top 10 Guest Origin Countries", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

## 7. Summary & Strategic Insights

1. **Lead Time Cancellation Spike**: Cancellation rates climb to over **45%** when lead times exceed 60 days, peaking at **54%** for 120+ days.
2. **Resort vs City Dynamics**: City Hotels face significantly higher cancellation rates (41.7% vs 27.8%) and higher baseline ADR, while Resort Hotels experience strong seasonal rate swings.
3. **OTA Dominance**: Online Travel Agents generate over **47%** of total bookings, but carry higher cancellation rates compared to direct bookings.